# W04A — Data Contract v1 + Calidad mínima (checks sí o sí)

**Objetivo:** formalizar un *contrato de datos* (mínimo viable) e implementar *checks* de calidad
que eviten problemas downstream (JOINs que inflan filas, métricas incoherentes, etc.).

**Recordemos que:**
- Un JOIN puede inflar filas si la clave no es única.
- Convierte esa lección en **política**: *antes* de transformar o unir, validamos.

**DDIA (Kleppmann):**
- Cap. 2: modelos de datos + “lo que prometemos” a consumidores.
- Cap. 4: evolución de esquema / compatibilidad: el contrato versiona expectativas.

In [1]:
import os
os.chdir("..")
os.getcwd()

'c:\\Users\\USUARIO WINDOWS\\CODIGOS\\PAZ CD'

In [2]:
# Setup común (cross-platform)
from pathlib import Path
import duckdb, json
from datetime import datetime, timezone

PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
DOCS_DIR = PROJECT_ROOT / "docs"
ART_DIR = PROJECT_ROOT / "artifacts"

RAW_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = RAW_DIR / "pscomppars.csv"

con = duckdb.connect(str(DB_PATH))

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

# Bronze: raw_ps (schema-on-read)
if not raw_csv.exists():
    raise FileNotFoundError(
        f"No encuentro {raw_csv}. "
        "Asegúrate de tener el CSV en data/raw/pscomppars.csv (de W01/W02)."
    )

raw_csv_abs = raw_csv.resolve()
con.execute(
    f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_quote(str(raw_csv_abs))})"
)

# Ver columnas disponibles
con.sql("DESCRIBE raw_ps").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

## 1) ¿Qué es un “data contract” (mínimo viable)?

Un **contrato de datos** es el *acuerdo explícito* entre productor y consumidor sobre:
- **Esquema** (nombres, tipos aproximados, significado)
- **Claves / granularidad** (qué identifica una fila)
- **Reglas de calidad** (nulos permitidos, rangos plausibles, unicidad, consistencia)
- **Versionado** (v1, v2…) y cómo cambian las expectativas

> En este curso lo tratamos como una “interfaz” entre etapas **Bronze → Silver → Gold**.

| Check                  | Tipo         | Qué valida                                                |
|------------------------|--------------|------------------------------------------------------------|
| pk_unique_pl_name      | uniqueness   | Que no existan duplicados en `pl_name`                    |
| nulls_pl_name          | completeness | Que `pl_name` no tenga valores nulos                      |
| nulls_hostname         | completeness | Que `hostname` no tenga valores nulos                     |
| disc_year_range        | validity     | Que el año de descubrimiento esté dentro de un rango válido |

El threshold es la cantidad máxima de errores que aceptas antes de decir “esto está mal”.

In [3]:
# DEMO (lectura guiada): contrato v1 (muy corto) como JSON para versionar expectativas
contract_v1 = {
  "dataset": "nasa_exoplanets_pscomppars",
  "version": "1.0.0",
  "table_bronze": "raw_ps",
  "grain": "1 fila ~ 1 planeta (pl_name) en catálogo",
  "primary_key_candidate": ["pl_name"],
  "required_columns": ["pl_name", "hostname"],
  "quality_checks": [
    {"name": "pk_unique_pl_name", "type": "uniqueness", "threshold": 0, "metric": "n_duplicates"},
    {"name": "nulls_pl_name", "type": "completeness", "threshold": 0, "metric": "n_nulls"},
    {"name": "nulls_hostname", "type": "completeness", "threshold": 0, "metric": "n_nulls"},
    {"name": "disc_year_range", "type": "validity", "threshold": 0, "metric": "n_out_of_range"},
  ],
  "notes": "Contrato v1: checks mínimos para no romper JOINs/agrupaciones."
}

(DOCS_DIR / "data_contract_v1.json").write_text(json.dumps(contract_v1, indent=2), encoding="utf-8")
print("Escribí docs/data_contract_v1.json")

Escribí docs/data_contract_v1.json


### Un contract más robusto sería

```json
contract_v2 = {
  "dataset": "nasa_exoplanets_pscomppars",
  "version": "1.0.0",
  "owner": "data-engineering@nasa.gov",
  "domain": "astronomia.exoplanetas",
  "table_bronze": "raw_ps",
  "grain": "1 fila = 1 planeta único identificado por pl_name",
  
  "schema_expected": {
    "pl_name": "string",
    "hostname": "string",
    "disc_year": "integer",
    "pl_orbper": "float",
    "pl_rade": "float"
  },

  "primary_key_candidate": ["pl_name"],
  "required_columns": ["pl_name", "hostname", "disc_year"],

  "quality_checks": [
    {
      "name": "pk_unique_pl_name",
      "type": "uniqueness",
      "metric": "n_duplicates",
      "threshold": 0,
      "severity": "critical"
    },
    {
      "name": "nulls_pl_name",
      "type": "completeness",
      "metric": "n_nulls",
      "threshold": 0,
      "severity": "critical"
    },
    {
      "name": "nulls_hostname",
      "type": "completeness",
      "metric": "n_nulls",
      "threshold": 0,
      "severity": "high"
    },
    {
      "name": "disc_year_range",
      "type": "validity",
      "metric": "n_out_of_range",
      "params": {"min": 1988, "max": 2025},
      "threshold": 0,
      "severity": "medium"
    },
    {
      "name": "orbital_period_positive",
      "type": "validity",
      "metric": "n_invalid",
      "rule": "pl_orbper > 0",
      "threshold": 0,
      "severity": "medium"
    }
  ],

  "business_rules": [
    "Cada planeta debe estar asociado a un único hostname.",
    "disc_year debe ser mayor o igual al año del primer exoplaneta detectado (1988)."
  ],

  "refresh_policy": {
    "frequency": "daily",
    "expected_arrival_time": "03:00 UTC",
    "late_data_tolerance_minutes": 120
  },

  "notes": "Contrato v2: incluye esquema, severidad, reglas de negocio y políticas de refresco."
}


## 2) Checks mínimos (la idea, antes del SQL)

**Completeness** (nulos en columnas clave):  
- `nulls_pl_name` = `COUNT(*) - COUNT(pl_name)`  
- `nulls_hostname` = `COUNT(*) - COUNT(hostname)`

**Uniqueness** (clave no debe duplicarse):  
- duplicados por `pl_name` → `GROUP BY pl_name HAVING COUNT(*)>1`

**Validity** (rangos plausibles):  
- `disc_year` debería estar en un rango razonable (ej. 1980–2026).

In [ ]:
# DEMO (docente): 3 checks con SQL básico

# 1) Nulos en columnas clave 
con.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(pl_name)  AS nulls_pl_name,
  COUNT(*) - COUNT(hostname) AS nulls_hostname
FROM raw_ps
""").show()

# 2) Duplicados por pl_name (si existen, alerta)
con.sql("""
SELECT pl_name, COUNT(*) AS c
FROM raw_ps
WHERE pl_name IS NOT NULL
GROUP BY pl_name
HAVING COUNT(*) > 1
ORDER BY c DESC
LIMIT 10
""").show()

# 3) Rangos plausibles de disc_year (ajusta si deseas)
con.sql("""
SELECT COUNT(*) AS n_out_of_range
FROM raw_ps
WHERE disc_year IS NOT NULL
  AND (disc_year < 1980 OR disc_year > 2026)
""").show()

## TU TURNO (en clase)

Vas a construir un mini-reporte de calidad para **12 columnas** (tú puedes ajustar la lista),
y luego lo guardas como tabla `quality_w03a` y como CSV en `artifacts/`.

### TU TURNO 1 — Nulos en 12 columnas clave

In [3]:
picked = [
  "pl_name", "hostname", "discoverymethod", "disc_year",
  "pl_orbper", "pl_rade", "pl_bmasse", "pl_eqt",
  "sy_dist", "ra", "dec", "st_teff"
]

parts = [
  f"SELECT '{c}' AS col, COUNT(*) - COUNT({c}) AS nulls FROM raw_ps"
  for c in picked
]

query = " UNION ALL ".join(parts) + " ORDER BY nulls DESC, col ASC"

quality_df = con.sql(query).df()

quality_df

,col,nulls
0,pl_eqt,1601
1,pl_orbper,340
2,st_teff,294
3,pl_rade,50
4,pl_bmasse,31
5,sy_dist,27
6,disc_year,1
7,dec,0
8,discoverymethod,0
9,hostname,0


In [4]:
con.register("quality_df_view", quality_df)

con.execute("""
CREATE OR REPLACE TABLE quality_w03a AS
SELECT * FROM quality_df_view
""")

print("Tabla quality_w03a creada en DuckDB")

Tabla quality_w03a creada en DuckDB


In [5]:
csv_path = ART_DIR / "quality_w03a.csv"

quality_df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"CSV guardado en: {csv_path}")

CSV guardado en: C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD\artifacts\quality_w03a.csv


### TU TURNO 2 — Un check de rango (elige 1)

In [7]:
query = """
SELECT
  COUNT(*) AS n_bad_pl_orbper
FROM raw_ps
WHERE pl_orbper IS NOT NULL
  AND pl_orbper <= 0
"""

range_df = con.sql(query).df()

range_df

,n_bad_pl_orbper
0,0


# W04B — Construyendo **Silver** (schema estable) + primeras vistas **Gold**

**Objetivo:** pasar de Bronze-lite (`raw_ps`) a un **Silver** con:
- columnas oficiales (Contract v1),
- llaves y granularidad explícitas,
- reglas simples de limpieza,
- dimensiones con **1 fila por clave** (para JOINs sanos),
y luego crear 2 vistas **Gold** de ejemplo.

In [ ]:
import os
os.chdir("..")
os.getcwd()

In [ ]:
# Setup (cross-platform)
from pathlib import Path
import duckdb, json
from datetime import datetime, timezone

PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
DOCS_DIR = PROJECT_ROOT / "docs"
ART_DIR  = PROJECT_ROOT / "artifacts"

RAW_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = RAW_DIR / "pscomppars.csv"
con = duckdb.connect(str(DB_PATH))

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

if not raw_csv.exists():
    raise FileNotFoundError(
        f"No encuentro {raw_csv}. Asegúrate de tenerlo en data/raw/pscomppars.csv (W01/W02)."
    )

raw_csv_abs = raw_csv.resolve()
con.execute(f'''
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_quote(str(raw_csv_abs))})
''')

con.sql("DESCRIBE raw_ps").show()

## 1) Diseño Silver (mínimo viable)

**Contract v1 (Core 16 columnas):**
`pl_name, hostname, discoverymethod, disc_year, sy_snum, sy_pnum, sy_dist, ra, dec, pl_orbper, pl_rade, pl_bmasse, pl_eqt, st_teff, st_rad, st_mass`

**Reglas Silver (hoy):**
- `pl_name` y `hostname` no nulos
- `disc_year` en [1980, 2026] si no es nulo
- rangos didácticos:
  - `pl_rade` > 0 y ≤ 30 si no es nulo
  - `pl_bmasse` > 0 si no es nulo

In [4]:
# TU TURNO 1: construir silver_planet
# Regla extra elegida: pl_orbper debe ser positivo si no es nulo

con.execute("DROP TABLE IF EXISTS silver_planet")

con.execute("""
CREATE TABLE silver_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  sy_snum,
  sy_pnum,
  sy_dist,
  ra,
  dec,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (disc_year IS NULL OR disc_year BETWEEN 1980 AND 2026)
  AND (pl_rade IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
  AND (pl_orbper IS NULL OR pl_orbper > 0)
""")

print("Tabla silver_planet creada")

con.sql("DESCRIBE silver_planet").show()

con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT pl_name) AS distinct_pl_name,
  COUNT(DISTINCT hostname) AS distinct_hostname
FROM silver_planet
""").show()

Tabla silver_planet creada
┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper

## 2) Dimensiones (1 fila por clave)

Estrategia “mesurada” (sin window functions):
- `GROUP BY hostname`
- para cada columna: `MAX(col)` para consolidar una fila por hostname

In [5]:
# TU TURNO 2: crear dim_host_full
# Columnas elegidas: sy_dist, ra, dec

con.execute("DROP TABLE IF EXISTS dim_host_full")

con.execute("""
CREATE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(sy_dist) AS sy_dist,
  MAX(ra) AS ra,
  MAX(dec) AS dec
FROM silver_planet
GROUP BY hostname
""")

print("Tabla dim_host_full creada")

con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT hostname) AS n_keys
FROM dim_host_full
""").show()

Tabla dim_host_full creada
┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4705 │   4705 │
└────────┴────────┘



## 3) Fact table (grain: 1 fila ≈ 1 planeta)

Creamos `fact_planet` desde Silver usando `SELECT DISTINCT`.
Si aparecen duplicados por `pl_name`, se documenta como issue de calidad.

In [6]:
# TU TURNO 3: crear fact_planet desde silver_planet usando DISTINCT

con.execute("DROP TABLE IF EXISTS fact_planet")

con.execute("""
CREATE TABLE fact_planet AS
SELECT DISTINCT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM silver_planet
""")

print("Tabla fact_planet creada")

con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT pl_name) AS n_pl
FROM fact_planet
""").show()

n_fact = con.execute("""
SELECT COUNT(*)
FROM fact_planet
""").fetchone()[0]

n_join = con.execute("""
SELECT COUNT(*)
FROM fact_planet f
JOIN dim_host_full h
  ON f.hostname = h.hostname
""").fetchone()[0]

print("n_fact:", n_fact)
print("n_join:", n_join)

Tabla fact_planet creada
┌────────┬───────┐
│ n_rows │ n_pl  │
│ int64  │ int64 │
├────────┼───────┤
│   6286 │  6286 │
└────────┴───────┘

n_fact: 6286
n_join: 6286


## 4) JOINs sanos + 2 vistas Gold
- JOIN sano: `COUNT(join)` ≈ `COUNT(fact)` (no inflar).
- Gold: `gold_by_method` y `gold_by_host`.

In [7]:
# TU TURNO 4: crear una vista Gold
# Opción elegida: gold_by_method

con.execute("DROP VIEW IF EXISTS gold_by_method")

con.execute("""
CREATE VIEW gold_by_method AS
SELECT
  discoverymethod,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 3) AS avg_radius_earth,
  ROUND(MEDIAN(pl_rade), 3) AS median_radius_earth,
  ROUND(AVG(pl_bmasse), 3) AS avg_mass_earth,
  ROUND(MEDIAN(pl_orbper), 3) AS median_orbital_period_days
FROM fact_planet
WHERE discoverymethod IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
""")

con.sql("SELECT * FROM gold_by_method LIMIT 10").show()

┌───────────────────────────────┬───────────┬──────────────────┬─────────────────────┬────────────────┬────────────────────────────┐
│        discoverymethod        │ n_planets │ avg_radius_earth │ median_radius_earth │ avg_mass_earth │ median_orbital_period_days │
│            varchar            │   int64   │      double      │       double        │     double     │           double           │
├───────────────────────────────┼───────────┼──────────────────┼─────────────────────┼────────────────┼────────────────────────────┤
│ Transit                       │      4650 │            4.342 │               2.448 │        120.782 │                      8.005 │
│ Radial Velocity               │      1181 │            9.763 │                12.6 │       1029.237 │                      298.2 │
│ Microlensing                  │       278 │            9.979 │                12.5 │         816.13 │                     3142.5 │
│ Imaging                       │        93 │            13.66 │     

## 5) Contract Silver v1 + trazabilidad

Escribimos `docs/data_contract_silver_v1.json` describiendo tablas Silver/Gold.
Si cambias columnas o reglas: incrementa versión.

In [8]:
gold_csv = ART_DIR / "gold_by_method.csv"

con.execute(
    f"""
    COPY (SELECT * FROM gold_by_method)
    TO {sql_quote(str(gold_csv))}
    WITH (HEADER, DELIMITER ',')
    """
)

print("Exporté:", gold_csv)

Exporté: C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD\artifacts\gold_by_method.csv


## Reflexión (bitácora)
- ¿Qué evidencia mínima te convence de que tu JOIN es sano?
- ¿Qué trade-off hay entre “limpiar mucho” vs “no perder datos”?

### TU TURNO 3 — Materializa un reporte (tabla) y expórtalo

In [16]:
from datetime import datetime, timezone

run_ts = datetime.now(timezone.utc).isoformat()

con.execute("DROP TABLE IF EXISTS quality_w03a")

sql = f"""
CREATE TABLE quality_w03a AS

SELECT
  {sql_quote(run_ts)} AS run_ts,
  'nulls_pl_name' AS check_name,
  'completeness' AS check_type,
  (COUNT(*) - COUNT(pl_name))::BIGINT AS metric_value
FROM raw_ps

UNION ALL

SELECT
  {sql_quote(run_ts)} AS run_ts,
  'nulls_hostname' AS check_name,
  'completeness' AS check_type,
  (COUNT(*) - COUNT(hostname))::BIGINT AS metric_value
FROM raw_ps

UNION ALL

SELECT
  {sql_quote(run_ts)} AS run_ts,
  'bad_pl_orbper' AS check_name,
  'validity_range' AS check_type,
  SUM(CASE WHEN pl_orbper IS NOT NULL AND pl_orbper <= 0 THEN 1 ELSE 0 END)::BIGINT AS metric_value
FROM raw_ps
"""

con.execute(sql)

con.sql("SELECT * FROM quality_w03a").show()

┌──────────────────────────────────┬────────────────┬────────────────┬──────────────┐
│              run_ts              │   check_name   │   check_type   │ metric_value │
│             varchar              │    varchar     │    varchar     │    int64     │
├──────────────────────────────────┼────────────────┼────────────────┼──────────────┤
│ 2026-05-27T17:49:17.876411+00:00 │ nulls_pl_name  │ completeness   │            0 │
│ 2026-05-27T17:49:17.876411+00:00 │ nulls_hostname │ completeness   │            0 │
│ 2026-05-27T17:49:17.876411+00:00 │ bad_pl_orbper  │ validity_range │            0 │
└──────────────────────────────────┴────────────────┴────────────────┴──────────────┘



In [17]:
ART_DIR.mkdir(parents=True, exist_ok=True)

out_csv = ART_DIR / f"w03a_quality_{run_ts.replace(':','-')}.csv"

con.execute(
    f"""
    COPY (SELECT * FROM quality_w03a)
    TO {sql_quote(str(out_csv))}
    WITH (HEADER, DELIMITER ',')
    """
)

print("Exporté:", out_csv)

Exporté: C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD\artifacts\w03a_quality_2026-05-27T17-49-17.876411+00-00.csv


In [ ]:
try:
    con.close()
    print("DuckDB connection closed.")
except NameError:
    print("No connection named 'con' in this notebook.")

## Para entregar (W04)
### En clase (evidencia mínima)
1) En `docs/w03a_quality_report.md` pega:
   - Output de **TU TURNO 1** (tabla de nulos en 12 columnas).
   - Output de **TU TURNO 2** (1 check de rango) + 2–3 líneas de interpretación.
2) En `docs/decisions_log.md`: 1 entrada corta:
   - “Qué 12 columnas escogí y por qué” + evidencia (tabla de nulos o conteos).
3) `docs/w03b_silver_report.md` con outputs:
   - `DESCRIBE silver_planet` + conteos (rows/distinct pl_name/hostname)
   - `dim_host_full`: `n_rows` vs `n_keys`
   - `n_fact` vs `n_join` (JOIN sano)
4) `docs/decisions_log.md`: 1 decisión:
   - “Reglas Silver aplicadas + evidencia”.

### Tarea (para la próxima clase)
1) Termina **TU TURNO 3**:
   - Explica en `docs/w03a_quality_report.md` con detalle el querry
   - Crear `quality_w03a` (3 checks mínimos) y exportar 1 CSV a `artifacts/`.
2) Completa TU TURNO 4 (una vista Gold) si no la hiciste.
3) Exporta 1 vista Gold a CSV en `artifacts/` (COPY ... TO).
4) (Opcional) si agregaste una regla extra, documéntala como versión `v1.0.1`.

## Reflexión (bitácora)
- ¿Qué check te sorprendió más y por qué?

El check que más me sorprendió fue el de nulos en las columnas principales, especialmente en variables físicas como `pl_orbper`, `pl_rade` y `pl_bmasse`. Aunque el dataset tiene identificadores completos como `pl_name` y `hostname`, no todos los planetas tienen disponibles sus parámetros físicos. Esto muestra que un dataset puede estar bien identificado, pero todavía tener vacíos importantes para el análisis científico.

- ¿Qué consecuencias tendría para W02B (JOINs) que `pl_name` o `hostname` tuviera muchos nulos/duplicados?

Si `pl_name` tuviera muchos nulos o duplicados, sería difícil mantener la granularidad de una fila por planeta. Esto afectaría la construcción de tablas como `fact_planet` y podría producir conteos incorrectos. Si `hostname` tuviera muchos nulos o duplicados, los JOINs de W02B con dimensiones de sistemas planetarios podrían fallar, perder filas o multiplicarlas accidentalmente. Por eso estos checks son importantes antes de construir capas Silver o hacer JOINs.


In [8]:
con.execute("DROP TABLE IF EXISTS silver_planet")

con.execute("""
CREATE TABLE silver_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  sy_snum,
  sy_pnum,
  sy_dist,
  ra,
  dec,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (disc_year IS NULL OR disc_year BETWEEN 1980 AND 2026)
  AND (pl_rade IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
  AND (pl_orbper IS NULL OR pl_orbper > 0)
""")

print("Tabla silver_planet creada")

Tabla silver_planet creada


In [9]:
con.sql("DESCRIBE silver_planet").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

In [10]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT pl_name) AS distinct_pl_name,
  COUNT(DISTINCT hostname) AS distinct_hostname
FROM silver_planet
""").show()

┌────────┬──────────────────┬───────────────────┐
│ n_rows │ distinct_pl_name │ distinct_hostname │
│ int64  │      int64       │       int64       │
├────────┼──────────────────┼───────────────────┤
│   6286 │             6286 │              4705 │
└────────┴──────────────────┴───────────────────┘



In [11]:
con.execute("DROP TABLE IF EXISTS dim_host_full")

con.execute("""
CREATE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(sy_dist) AS sy_dist,
  MAX(ra) AS ra,
  MAX(dec) AS dec
FROM silver_planet
GROUP BY hostname
""")

print("Tabla dim_host_full creada")

Tabla dim_host_full creada


In [12]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT hostname) AS n_keys
FROM dim_host_full
""").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4705 │   4705 │
└────────┴────────┘



In [14]:
con.execute("DROP TABLE IF EXISTS fact_planet")

con.execute("""
CREATE TABLE fact_planet AS
SELECT DISTINCT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM silver_planet
""")

print("Tabla fact_planet creada")

Tabla fact_planet creada


In [15]:
n_fact = con.execute("""
SELECT COUNT(*) FROM fact_planet
""").fetchone()[0]

n_join = con.execute("""
SELECT COUNT(*)
FROM fact_planet f
JOIN dim_host_full h
  ON f.hostname = h.hostname
""").fetchone()[0]

print("n_fact:", n_fact)
print("n_join:", n_join)

n_fact: 6286
n_join: 6286
